# This notebook first defines GEVs then smaples from them and    
# then uses MLE to calculate the percentiles and computes the bias
##### Author: Omid Emamjomehzadeh (https://www.omidemam.com/)
##### Supervisor: Dr. Omar Wani (https://engineering.nyu.edu/faculty/omar-wani)
##### Hydrologic Systems Group @NYU (https://www.omarwani.com/)

In [ ]:
#import libraries
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib as mpl
import pandas as pd
import numpy as np
import pymc3 as pm3
import scipy.stats as stats
from scipy import stats
from scipy.stats import genextreme
from scipy.stats import norm, gaussian_kde
from scipy.optimize import minimize
import seaborn as sns
import arviz as az
import pymc3 as pm
import os
import theano.tensor as tt
from theano.compile.ops import as_op
import theano
import pandas as pd
from pymc3.distributions.dist_math import bound
from scipy.stats import genextreme
import warnings
from arviz.plots import plot_utils as azpu
import arviz as az
from tqdm import tqdm
%matplotlib inline
sns.set()
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore', UserWarning)
theano.config.warn.round=False
from watermark import watermark
from datetime import datetime
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt
plt.style.use('default')   # <-- resets to plain Matplotlib look
from itertools import chain
import random


In [5]:
# read parameters
mle_1h = r"D:\BMM-IDF4Drainage_data_results\Percentile\gev_params_1h_mle_AORC.csv"
# Read the path
df_mle_1h = pd.read_csv(mle_1h)
df_mle_1h

,City,shape(c),loc,scale
0,Birmingham,0.072627,22.491624,10.228208
1,Phoenix,-0.267678,8.175461,3.664874
2,Little Rock,-0.055021,21.988082,9.258394
3,Los Angeles,0.042387,11.861270,5.693252
4,Denver,-0.413798,7.283690,4.342796
5,Hartford,-0.285448,14.298267,6.394461
6,Wilmington,-0.035867,18.485770,8.115728
7,Miami,-0.285025,19.471994,9.999978
8,Atlanta,-0.226416,16.665304,7.189982
9,Boise,-0.174572,5.606417,2.189396


# AORC noise
The AORC precipitation values are related to the true (observed) precipitation by

$$
P_{\text{AORC}} = P \times MB
$$

where $MB$ (multiplicative bias) follows a lognormal distribution:

$$
MB \sim \text{LogNormal}\left(\mu = \ln(GM), \; \sigma = \sqrt{\ln\!\left(1 + (GCV/100)^2\right)}\right)
$$

**Table A1 — AORC (CONUS) Statistics**

| Year       | GM   | GCV (%) |
|-------------|------|----------|
| 1979–2001   | 0.98 | 1.8      |
| 2002–2009   | 0.98 | 2.4      |
| 2010–2015   | 1.00 | 3.7      |
| 2016–2021   | 1.00 | 2.9      |

In [16]:
def add_aorc_noise(values, start_year=1979, end_year=2024, seed=None):
    """
    Adds AORC-style multiplicative lognormal noise to rainfall values 
    for the CONUS domain, using year-based GM and GCV statistics.

    Parameters
    ----------
    values : np.ndarray
        1D array of rainfall values (first element = start_year, last = end_year).
    start_year : int, optional
        First year of the dataset. Default is 1979.
    end_year : int, optional
        Last year of the dataset. Default is 2024.
    seed : int or None, optional
        Random seed for reproducibility.

    Returns
    -------
    noisy_values : np.ndarray
        Array of rainfall values with multiplicative AORC noise applied.
    """

    # --- Check inputs ---
    n_years = end_year - start_year + 1
    if len(values) != n_years:
        raise ValueError(f"Expected {n_years} values (one per year), got {len(values)}.")

    # --- AORC CONUS GM and GCV table (Table A1) ---
    eras = [
        (1800, 2001, 0.98, 1.8),
        (2002, 2009, 0.98, 2.4),
        (2010, 2015, 1.00, 3.7),
        (2016, 2021, 1.00, 2.9),
        (2022, 2100, 1.00, 2.9),  # extrapolate last period
    ]

    rng = np.random.default_rng(seed)
    noisy_values = np.zeros_like(values, dtype=float)

    # --- Loop over each year and apply lognormal noise ---
    for i, year in enumerate(range(start_year, end_year + 1)):
        # Get GM and GCV for that year
        for y0, y1, GM, GCV in eras:
            if y0 <= year <= y1:
                mu = np.log(GM)
                sigma = np.sqrt(np.log(1 + (GCV / 100.0) ** 2))
                MB = rng.lognormal(mean=mu, sigma=sigma)  # multiplicative bias
                noisy_values[i] = values[i] * MB
                break

    return noisy_values,MB

In [10]:
# Define GEV log-likelihood function
def gev_loglik(theta, data):
    c, loc, scale = theta
    gev = genextreme(c=c, loc=loc, scale=scale)
    return -np.sum(np.log(gev.pdf(data)))

In [ ]:
results = []
simseries_accum = []
# --- settings ---
n_sims = 500
random.seed(12345)
seeds = random.sample(range(1, 100001), n_sims)

T = np.array([2, 5, 10, 25, 50, 100])
p = 1 - 1 / T

sample_sizes = [46]   # last N years
AORC_END_YEAR = 2024              # "last N years" ends at 2024


for idx, row in df_mle_1h.iterrows():
    city       = row['City']
    c_true     = row['shape(c)']   # keep sign convention consistent with your genextreme usage
    loc_true   = row['loc']
    scale_true = row['scale']

    # true RLs once per city
    rl_true = genextreme.ppf(p, c=c_true, loc=loc_true, scale=scale_true)

    for sample_size in sample_sizes:

        # pre-alloc PER (CITY, SAMPLE_SIZE)
        bias   = np.zeros((n_sims, len(T)))
        means  = np.zeros_like(bias)
        ci_low = np.zeros_like(bias)
        ci_high= np.zeros_like(bias)

        n_done = n_sims  # will drop if converged early

        # year window for "last N years"
        end_year = AORC_END_YEAR
        start_year = end_year - sample_size + 1

        for i in range(1, n_sims + 1):  # 1-based loop
            end_year = AORC_END_YEAR
            start_year = end_year - sample_size + 1
            # --- simulate one replicate with N = sample_size ---
            samples = genextreme.rvs(c=c_true, loc=loc_true, scale=scale_true, size=sample_size, random_state=seeds[i-1])
            obs_samples = add_aorc_noise(samples, start_year=start_year, end_year=end_year,seed=seeds[i-1])

            # MLE fit (robust routine with fallback)
            c_mle, loc_mle, scale_mle = genextreme.fit(obs_samples)

            # MLE RLs and bias for this replicate
            rl_mle = genextreme.ppf(p, c=c_mle, loc=loc_mle, scale=scale_mle)
            bias[i-1, :] = rl_mle - rl_true     # << fixed off-by-one

            # --- running mean & 95% CI ---
            m = bias[:i].mean(axis=0)
            if i > 1:
                se = bias[:i].std(axis=0, ddof=1) / np.sqrt(i)
            else:
                se = np.full(len(T), np.inf)

            means[i-1]  = m
            hw          = 1.96 * se
            ci_low[i-1] = m - hw
            ci_high[i-1]= m + hw

        # --- plotting with actual length ---
        # --- plotting with actual length ---
        x = np.arange(1, n_done + 1)

        fig, ax = plt.subplots(figsize=(5, 5))
        ax2 = ax.twinx()  # right axis for per-simulation dots

        line_handles = []
        for j, t in enumerate(T):
            # mean line on LEFT axis
            h, = ax.plot(x, means[:, j], label=f'{t}-yr', lw=1,zorder=1)
            line_handles.append(h)
            # CI band on LEFT axis (match color)
            ax.fill_between(x, ci_low[:, j], ci_high[:, j], alpha=0.15, color=h.get_color(),zorder=2)
            # per-simulation dots on RIGHT axis in SAME color (no legend)
            ax2.plot(x, bias[:n_done, j], '.', alpha=0.3, ms=2, color=h.get_color(), label='_nolegend_', zorder=1)

        # zero lines on both axes for clarity
        ax.axhline(0, color='k', lw=1, linestyle='--', alpha=0.6)
        ax2.axhline(0, color='k', lw=1, linestyle='--', alpha=0.3)

        ax.set_xlabel('Number of simulations')
        ax.set_ylabel('Average bias (mm)')                 # left axis (lines/CI)
        ax2.set_ylabel('Per-simulation bias (mm)')         # right axis (scatter)
        ax.set_ylim(-10, 10)
        ax.set_xlim(1, n_done)
        ax2.set_xlim(1, n_done)  # keep x synced

        ax.set_title(
            f'Mean bias and 95% CI of MLE-estimated GEV return levels\n'
            f'(MLE 1hr parameters, {city}, sample size N={sample_size}+Noise)'
        )

        # legend only for the mean lines
        ax.legend(loc='upper center',
                  handlelength=1.0, handletextpad=0.4, columnspacing=0.8,
                  bbox_to_anchor=(0.5, -0.1),
                  ncol=6, frameon=False)

        ax.grid(True, linestyle="--", alpha=0.6)
        fig.tight_layout()

        fig_path = rf"D:\BMM-IDF4Drainage_data_results\Bias\MLE\Figures\Noise_stationary\Bias_MLE_AORC_Conv_1hr_{city}_N{sample_size}.png"
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.show()

        # --- save per-simulation bias series with n_done ---
        sim_idx = np.arange(1, n_done + 1)
        df_bias_city = pd.DataFrame({'City': city, 'sample_size': sample_size, 'sim': sim_idx})
        for j, t in enumerate(T):
            df_bias_city[f'bias_T{int(t)}'] = bias[:, j]
        series_path = rf"D:\BMM-IDF4Drainage_data_results\Bias\MLE\bias_series\Noise_stationary\bias_sim_series_1hr_MLE_{city}_N{sample_size}.csv"
        df_bias_city.to_csv(series_path, index=False)

        # --- final summary (example: last row stats) ---
        mean_final   = means[-1]
        ci_low_final = ci_low[-1]
        ci_high_final= ci_high[-1]

        # --- append summary for this (city, N) ---
        rec = {'City': city, 'sample_size': sample_size, 'n_sims_final': n_done}
        for j, t in enumerate(T):
            rec[f'mean_T{int(t)}'] = float(mean_final[j])
            rec[f'ci_low_T{int(t)}'] = float(ci_low_final[j])
            rec[f'ci_high_T{int(t)}'] = float(ci_high_final[j])
        results.append(rec)

        # --- make cumulative DataFrame and save (grows across sizes & cities) ---
        df_progress = pd.DataFrame(results)

        # organize columns nicely
        ordered_cols = ['City', 'sample_size', 'n_sims_final'] + list(chain.from_iterable(
            [[f'mean_T{int(t)}', f'ci_low_T{int(t)}', f'ci_high_T{int(t)}'] for t in T]
        ))
        df_progress = df_progress.reindex(columns=ordered_cols)

        # save cumulative results file
        latest_path = r"D:\BMM-IDF4Drainage_data_results\Bias\MLE\bias_MLE_AORC_1hr_summary_noisystationary.csv"
        df_progress.to_csv(latest_path, index=False)

In [ ]:
#Getting the current date and time
current_datetime = datetime.now()

# Printing the date and time
print("Date and Time of the Notebook Analysis:", current_datetime)

Date and Time of the Notebook Analysis: 2024-07-29 17:22:56.044172


In [ ]:
%load_ext watermark

# Print the Python version and some dependencies
%watermark -v -m -p numpy,pandas,matplotlib,pymc3,scipy,seaborn,arviz,os,theano,warnings,tqdm,watermark


Python implementation: CPython
Python version       : 3.9.19
IPython version      : 8.18.1

numpy     : 1.22.1
pandas    : 2.0.3
matplotlib: 3.8.4
pymc3     : 3.11.6
scipy     : 1.7.3
seaborn   : 0.13.2
arviz     : 0.12.1
os        : unknown
theano    : 1.1.2
warnings  : unknown
tqdm      : 4.66.4
watermark : 2.4.3

Compiler    : MSC v.1929 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 24
Architecture: 64bit

